In [1]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_excel('FINA4359_SPMO.xlsx', sheet_name='Sheet1')
df['ret'] = (df['SPMO'] / df['SPMO'].shift(1) - 1) * 100
df = df.dropna(subset=['ret'])
df['excess_ret'] = df['ret'] - df['RF']

X = sm.add_constant(df[['Mkt-RF', 'MOM']])
y = df['excess_ret']
model = sm.OLS(y, X).fit()
print("Q1")
print("───────────────────────────────────────────────────────────────")
print("Two-factor regression: SPMO excess return ~ Mkt-RF + MOM")
print(f"Alpha (const)         : {model.params['const']:9.4f}   t = {model.tvalues['const']:6.2f}   p = {model.pvalues['const']:6.4f}")
print(f"Mkt-RF               : {model.params['Mkt-RF']:9.4f}   t = {model.tvalues['Mkt-RF']:6.2f}   p = {model.pvalues['Mkt-RF']:6.4f}")
print(f"Beta MOM             : {model.params['MOM']:.4f}       t = {model.tvalues['MOM']:6.2f}   p = {model.pvalues['MOM']:6.4f}\n")

X = df[['Mkt-RF', 'SMB', 'HML', 'MOM']]
X = sm.add_constant(X)
y = df['excess_ret']

model = sm.OLS(y, X).fit()
print("Q4")
print("───────────────────────────────────────────────────────────────")
print("Four-factor regression: SPMO excess return ~ Mkt-RF + SMB + HML + MOM")
print("───────────────────────────────────────────────────────────────")
print(f"Observations          : {len(df):3d}")
print(f"R-squared             : {model.rsquared:8.4f}")
print(f"Adjusted R-squared    : {model.rsquared_adj:8.4f}")
print("\nCoefficients:")
print("───────────────────────────────────────────────────────────────")
print(f"Alpha (const)         : {model.params['const']:9.4f}   t = {model.tvalues['const']:6.2f}   p = {model.pvalues['const']:6.4f}")
print(f"Mkt-RF                : {model.params['Mkt-RF']:9.4f}   t = {model.tvalues['Mkt-RF']:6.2f}   p = {model.pvalues['Mkt-RF']:6.4f}")
print(f"SMB                   : {model.params['SMB']:9.4f}   t = {model.tvalues['SMB']:6.2f}   p = {model.pvalues['SMB']:6.4f}")
print(f"HML                   : {model.params['HML']:9.4f}   t = {model.tvalues['HML']:6.2f}   p = {model.pvalues['HML']:6.4f}")
print(f"MOM                   : {model.params['MOM']:9.4f}   t = {model.tvalues['MOM']:6.2f}   p = {model.pvalues['MOM']:6.4f}")
print("───────────────────────────────────────────────────────────────")

Q1
───────────────────────────────────────────────────────────────
Two-factor regression: SPMO excess return ~ Mkt-RF + MOM
Alpha (const)         :    0.0581   t =   0.37   p = 0.7100
Mkt-RF               :    1.0435   t =  28.81   p = 0.0000
Beta MOM             : 0.4314       t =  10.04   p = 0.0000

Q4
───────────────────────────────────────────────────────────────
Four-factor regression: SPMO excess return ~ Mkt-RF + SMB + HML + MOM
───────────────────────────────────────────────────────────────
Observations          : 121
R-squared             :   0.8819
Adjusted R-squared    :   0.8779

Coefficients:
───────────────────────────────────────────────────────────────
Alpha (const)         :    0.0127   t =   0.08   p = 0.9345
Mkt-RF                :    1.0655   t =  28.89   p = 0.0000
SMB                   :   -0.1465   t =  -2.47   p = 0.0151
HML                   :    0.0346   t =   0.81   p = 0.4181
MOM                   :    0.3992   t =   8.54   p = 0.0000
──────────────────────

In [2]:
import pandas as pd
import statsmodels.api as sm

# Load the data from Excel
df = pd.read_excel('FINA4359_JMOM_MTUM.xlsx', sheet_name='Sheet1')

# Calculate monthly returns (%) for JMOM and MTUM
df['ret_JMOM'] = (df['JMOM'] / df['JMOM'].shift(1) - 1) * 100
df['ret_MTUM'] = (df['MTUM'] / df['MTUM'].shift(1) - 1) * 100

# Calculate excess returns
df['excess_JMOM'] = df['ret_JMOM'] - df['RF']
df['excess_MTUM'] = df['ret_MTUM'] - df['RF']

# Drop rows with NaN (first row and any missing data)
df = df.dropna(subset=['excess_JMOM', 'excess_MTUM', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM'])

# Define factors
factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']

# Function to run regression
def run_regression(y, X, hac_lags=3):
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': hac_lags})
    return model

# JMOM regression
X = df[factors]
model_jmom = run_regression(df['excess_JMOM'], X)

# MTUM regression
model_mtum = run_regression(df['excess_MTUM'], X)

# Print JMOM results
print("JMOM Regression Results")
print(model_jmom.summary())

# Print MTUM results
print("\nMTUM Regression Results")
print(model_mtum.summary())

JMOM Regression Results
                            OLS Regression Results                            
Dep. Variable:            excess_JMOM   R-squared:                       0.968
Model:                            OLS   Adj. R-squared:                  0.966
Method:                 Least Squares   F-statistic:                     596.3
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           3.58e-70
Time:                        05:09:23   Log-Likelihood:                -127.39
No. Observations:                  97   AIC:                             268.8
Df Residuals:                      90   BIC:                             286.8
Df Model:                           6                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1336      0